# CT-ScanEval: run a bigger Gemini model (free tier)

Uses the same Colab Secrets as before (`GEMINI_API_KEY`, `GITHUB_TOKEN`). Just use **Runtime, Run all**.

It runs **Gemini 3.6 Flash**. If that gives no results (for example, not available on the free tier), it tries **Gemini 3.8 Flash**. Then it pushes the results to GitHub. It takes about 5 minutes.

In [ ]:
!git clone -q https://github.com/Moyo8/-CT-ScanEval /content/CT-ScanEval-run2
%cd /content/CT-ScanEval-run2
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('Gemini key loaded')

In [ ]:
!python run_eval.py --provider gemini --model gemini-3.6-flash --trials 3 --pace 7

In [ ]:
import glob
if not glob.glob('results/*__gemini__gemini-3.6-flash/results.jsonl'):
    print('Gemini 3.6 Flash gave no results - trying Gemini 3.8 Flash')
    get_ipython().system('python run_eval.py --provider gemini --model gemini-3.8-flash --trials 3 --pace 7')
runs = glob.glob('results/*__gemini__gemini-3.[68]-flash/run.json')
print('NEW RESULTS:', runs if runs else 'none - copy the error messages above and send them to Claude')

In [ ]:
import base64, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
def git(*args):
    r = subprocess.run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***').replace(auth, '***').strip())
    return r.returncode
git('config', 'user.name', 'Moyo8')
git('config', 'user.email', 'Moyo8@users.noreply.github.com')
git('add', 'results', 'transcripts')
git('commit', '-m', 'Bigger Gemini model run from Colab')
git('pull', '--rebase', 'origin', 'main')
print('pushed OK' if git('push', 'origin', 'HEAD:main') == 0 else 'PUSH FAILED - check the token')